## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [1]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [2]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [3]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [4]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [5]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [6]:
# Write collection_bbox(features) and apply to all four LOD files
def collection_bbox(features):
    all_lons = [c[0] for f in features for c in f["geometry"]["coordinates"]]
    all_lats = [c[1] for f in features for c in f["geometry"]["coordinates"]]
    return [min(all_lons), min(all_lats), max(all_lons), max(all_lats)]

lod_dir = Path("../../data/lod")
lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print(f"{'Level':<12}  {'lon_min':>9} {'lat_min':>9} {'lon_max':>9} {'lat_max':>9}")
print("-" * 52)
for name, filename in lod_files.items():
    with open(lod_dir / filename) as f:
        fc = json.load(f)
    bb = collection_bbox(fc["features"])
    print(f"{name:<12}  {bb[0]:>9.4f} {bb[1]:>9.4f} {bb[2]:>9.4f} {bb[3]:>9.4f}")

# The coarse level has a noticeably smaller extent than the other three. The
# scalerank filter removes features in some regions entirely, so those areas
# drop out of the collection bbox. Medium, fine, and extra_fine all cover the
# same global extent.

Level           lon_min   lat_min   lon_max   lat_max
----------------------------------------------------
coarse        -123.0147  -41.4752  150.9617   60.9765
medium        -150.1122  -51.8947  179.3578   69.6044
fine          -150.1122  -51.8947  179.3578   69.6044
extra_fine    -150.1122  -51.8953  179.3578   69.6044


## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [7]:
# Find the 5 features with the largest bounding box area in railroads_fine.geojson
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

def bbox_area(feature):
    bb = feature_bbox(feature)
    return (bb[2] - bb[0]) * (bb[3] - bb[1])

top5 = sorted(fine["features"], key=bbox_area, reverse=True)[:5]

print(f"{'Rank':<6} {'Area':>8}  {'Category':>10}  {'Continent':<15}  bbox")
print("-" * 70)
for i, feat in enumerate(top5, 1):
    bb   = feature_bbox(feat)
    area = bbox_area(feat)
    cat  = feat["properties"].get("category")
    cont = feat["properties"].get("continent", "")
    print(f"{i:<6} {area:>8.4f}  {str(cat):>10}  {cont:<15}  {[round(v, 2) for v in bb]}")

# These results make geographic sense. The largest bbox features are long-distance
# lines that cross mountainous or sparse terrain, producing wide geographic extents
# relative to their actual track length. Asia and Oceania dominate because their
# rail lines span large, sparsely-populated regions.

Rank       Area    Category  Continent        bbox
----------------------------------------------------------------------
1       29.3123           0  Asia             [90.61, 29.65, 94.94, 36.41]
2       18.6487           0  Oceania          [132.26, -23.55, 134.32, -14.53]
3       17.7045           3  South America    [-64.09, -26.19, -58.16, -23.21]
4       17.3866           2  Europe           [14.02, 54.8, 20.0, 57.71]
5       12.3547           2  South America    [-49.14, -5.49, -44.37, -2.9]


## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

---

In [8]:
# Scenario: two diagonal lines crossing the same corner points, one running
# northeast to southwest, another running northwest to southeast. Both share
# the same lon_min, lat_min, lon_max, lat_max but trace completely different paths.
#
# No, it does not cause a problem. Bounding box culling is a conservative test.
# A false positive (bbox overlaps viewport but geometry doesn't) means a feature
# gets sent to the renderer unnecessarily, which wastes a small amount of work.
# A false negative is impossible. If a feature's geometry intersects the viewport,
# its bbox must also intersect it. Correctness is guaranteed. Only efficiency
# is slightly reduced by features that share bounding boxes with the viewport.

## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.